# 07 - QRM pre-check: HelpSteer2 axis decorrelation

Project context: this repository supports a master's thesis on Preference-Aware Coefficient Correction for Rewarded-Soups-style model merging. RQ2 asks whether parameter geometry `R` can proxy reward behavior. ArmoRM is the strictly read-only evaluation model and must not become a training signal.

For a possible PPO path, a held-out reward model different from ArmoRM is needed. This notebook tests whether `nicolinho/QRM-Llama3.1-8B` is a plausible held-out RM by measuring whether its five HelpSteer2 objective scores are less co-rated than ArmoRM/HelpSteer2, especially the known help/correctness coupling. It trains nothing.

External model-card facts used here: QRM reports a scalar `output.score`, distributional `output.reward_quantiles`, and per-objective expectations in `output.rewards`; the first five objectives are `helpsteer-helpfulness`, `helpsteer-correctness`, `helpsteer-coherence`, `helpsteer-complexity`, `helpsteer-verbosity`. The QRM model card also states that it uses `Skywork/Skywork-Reward-Llama-3.1-8B` as backbone and Skywork reward preference data for the gating network.

Goal in one sentence: Entscheidet ohne einen einzigen Trainingslauf, ob QRM als held-out-Reward die help/correctness-Kollinearitaet durchbricht - und damit, ob der PPO-Pfad ueberhaupt eine Chance auf echte Konfliktstruktur in R hat.

## Setup

Clone or update the repository, install the lightweight dependencies, set deterministic seeds, and print the full configuration. This block does not load ArmoRM and does not train anything.

Restart the runtime once after this install cell if Colab already imported a different Transformers version. Then continue from this notebook from the top.


In [ ]:
%cd /content

import os
import json
import math
import random
import shutil
import subprocess
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

repo_path = Path('/content/master-thesis')
repo_url = 'https://github.com/NZhang137/master-thesis.git'

if (repo_path / '.git').is_dir():
    print('[A] Repository exists; pulling latest changes.')
    subprocess.run(['git', '-C', str(repo_path), 'pull', '--ff-only'], check=False)
else:
    print('[A] Repository missing; cloning from GitHub.')
    if repo_path.exists():
        shutil.rmtree(repo_path)
    subprocess.run(['git', 'clone', repo_url, str(repo_path)], check=True)

%cd /content/master-thesis

!pip install -q "transformers==4.45.2" datasets accelerate scipy numpy

import numpy as np

try:
    import torch
except Exception as error:
    torch = None
    print(f'[A] Torch import failed for now: {error}')

CONFIG = {
    'QRM_MODEL': 'nicolinho/QRM-Llama3.1-8B',
    'N_SAMPLES': 500,
    'HS2_SPLIT': 'validation',
    'SEED': 137,
    'BATCH_SIZE': 2,
    'MAX_LENGTH': 4096,
    'DECORR_THRESHOLD': 0.7,
    'HS2_HELP_CORR_REFERENCE': 0.943,
    'OUTPUT_DIR': 'results/qrm_precheck_axis_decorrelation',
    'OUTPUT_ZIP': 'qrm_precheck_axis_decorrelation_outputs.zip',
    'ARMORM_SCORE_PATH': '',
}

random.seed(CONFIG['SEED'])
np.random.seed(CONFIG['SEED'])
if torch is not None:
    torch.manual_seed(CONFIG['SEED'])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(CONFIG['SEED'])

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = (PROJECT_ROOT / CONFIG['OUTPUT_DIR']).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('[A] Project root:', PROJECT_ROOT)
print('[A] Output dir:', OUTPUT_DIR)
print('[A] Config:')
print(json.dumps(CONFIG, indent=2, sort_keys=True))

assert CONFIG['QRM_MODEL'] != 'RLHFlow/ArmoRM-Llama3-8B-v0.1', 'QRM_MODEL must not be ArmoRM.'
assert CONFIG['N_SAMPLES'] > 0, 'N_SAMPLES must be positive.'
assert 0.0 < CONFIG['DECORR_THRESHOLD'] < 1.0, 'DECORR_THRESHOLD should be a correlation threshold in (0, 1).'
print('[A] Safety: ArmoRM is not used as a training signal; this notebook trains nothing.')

## Load HelpSteer2 samples

Load the HelpSteer2 validation split, sample rows deterministically, keep the five ground-truth ratings, and render each prompt/response pair with the QRM chat template.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

ATTRIBUTES = ['helpfulness', 'correctness', 'coherence', 'complexity', 'verbosity']
QRM_HELPSTEER_OBJECTIVES = [
    'helpsteer-helpfulness',
    'helpsteer-correctness',
    'helpsteer-coherence',
    'helpsteer-complexity',
    'helpsteer-verbosity',
]

print(f"[B] Loading HelpSteer2 split: {CONFIG['HS2_SPLIT']}")
dataset = load_dataset('nvidia/HelpSteer2', split=CONFIG['HS2_SPLIT'])
print('[B] Dataset rows:', len(dataset))
print('[B] Dataset columns:', dataset.column_names)

required_columns = set(['prompt', 'response', *ATTRIBUTES])
missing_columns = sorted(required_columns - set(dataset.column_names))
assert not missing_columns, f'Missing HelpSteer2 columns: {missing_columns}'
assert len(dataset) >= CONFIG['N_SAMPLES'], f"Need {CONFIG['N_SAMPLES']} samples, but split has only {len(dataset)} rows."

rng = np.random.default_rng(CONFIG['SEED'])
sample_indices = rng.choice(len(dataset), size=CONFIG['N_SAMPLES'], replace=False)

print('[B] Loading QRM tokenizer for apply_chat_template rendering.')
qrm_tokenizer = AutoTokenizer.from_pretrained(CONFIG['QRM_MODEL'], use_fast=True, trust_remote_code=True)
if qrm_tokenizer.pad_token_id is None:
    qrm_tokenizer.pad_token = qrm_tokenizer.eos_token
    print('[B] Tokenizer had no pad token; using eos token as pad token.')

records = []
chat_texts = []
hs2_ratings = []

for output_index, dataset_index in enumerate(sample_indices):
    row = dataset[int(dataset_index)]
    ratings = [float(row[attribute]) for attribute in ATTRIBUTES]
    messages = [
        {'role': 'user', 'content': str(row['prompt'])},
        {'role': 'assistant', 'content': str(row['response'])},
    ]
    chat_text = qrm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    assert isinstance(chat_text, str) and chat_text.strip(), 'Rendered chat text is empty.'
    records.append({
        'output_index': output_index,
        'dataset_index': int(dataset_index),
        'prompt': str(row['prompt']),
        'response': str(row['response']),
        'ratings': dict(zip(ATTRIBUTES, ratings)),
    })
    chat_texts.append(chat_text)
    hs2_ratings.append(ratings)

hs2_ratings = np.asarray(hs2_ratings, dtype=np.float64)
sample_indices = np.asarray(sample_indices, dtype=np.int64)

assert hs2_ratings.shape == (CONFIG['N_SAMPLES'], len(ATTRIBUTES)), hs2_ratings.shape
assert np.all(np.isfinite(hs2_ratings)), 'HelpSteer2 ratings contain non-finite values.'
assert len(chat_texts) == CONFIG['N_SAMPLES'], 'chat_texts length mismatch.'

records_path = OUTPUT_DIR / 'sample_records.jsonl'
with records_path.open('w', encoding='utf-8') as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f'[B] Sampled {len(records)} rows with seed {CONFIG["SEED"]}.')
print('[B] First sampled dataset indices:', sample_indices[:10].tolist())
print('[B] Saved sample metadata:', records_path)
print('[B] Rating means:', dict(zip(ATTRIBUTES, np.round(hs2_ratings.mean(axis=0), 3))))

## Load and score QRM

Load `nicolinho/QRM-Llama3.1-8B` with `trust_remote_code=True` and score the rendered samples in batches. The notebook extracts the five HelpSteer2 objective expectations from `output.rewards`, not just the aggregate scalar `output.score`.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

assert torch.cuda.is_available(), 'QRM-8B needs a CUDA runtime. In Colab, use an A100/High-RAM runtime if possible.'
print('[C] CUDA device:', torch.cuda.get_device_name(0))
print('[C] Loading QRM model. OOM hint: QRM-8B is about 15GB on disk and generally needs a large GPU runtime.')

load_kwargs = {
    'trust_remote_code': True,
    'torch_dtype': torch.bfloat16,
    'device_map': 'cuda',
}

try:
    qrm_model = AutoModelForSequenceClassification.from_pretrained(CONFIG['QRM_MODEL'], **load_kwargs)
except RuntimeError as error:
    message = str(error)
    print('[C] QRM load failed with RuntimeError.')
    if 'out of memory' in message.lower() or 'cuda' in message.lower():
        print('[C] OOM/CUDA hint: switch to A100/High-RAM, reduce BATCH_SIZE, or restart the runtime before retrying.')
    raise
except Exception as error:
    print('[C] Primary load with device_map="cuda" failed; retrying with device_map={"": 0}.')
    print('[C] Original error:', repr(error))
    fallback_kwargs = dict(load_kwargs)
    fallback_kwargs['device_map'] = {'': 0}
    qrm_model = AutoModelForSequenceClassification.from_pretrained(CONFIG['QRM_MODEL'], **fallback_kwargs)

qrm_model.eval()
model_device = next(qrm_model.parameters()).device
print('[C] Model first-parameter device:', model_device)
print('[C] Model num_objectives:', getattr(qrm_model.config, 'num_objectives', 'unknown'))
print('[C] Model num_quantiles:', getattr(qrm_model.config, 'num_quantiles', 'unknown'))

# QRM custom output notes:
# - output.score is the gated scalar expectation over objectives.
# - output.reward_quantiles is the gated reward distribution over quantiles.
# - output.rewards is the per-objective expected reward matrix; by model-card order,
#   columns 0..4 are the five HelpSteer2 objectives listed in QRM_HELPSTEER_OBJECTIVES.
def extract_qrm_helpsteer2_scores(output, expected_batch_size: int) -> np.ndarray:
    assert hasattr(output, 'score'), 'QRM output is missing aggregate score.'
    assert hasattr(output, 'reward_quantiles'), 'QRM output is missing reward_quantiles.'
    assert hasattr(output, 'rewards'), 'QRM output is missing per-objective rewards.'
    rewards = output.rewards.detach().float().cpu()
    if rewards.ndim == 1:
        rewards = rewards.unsqueeze(0)
    assert rewards.ndim == 2, f'Expected output.rewards to be 2D, got {tuple(rewards.shape)}.'
    assert rewards.shape[0] == expected_batch_size, f'Batch mismatch: {rewards.shape[0]} vs {expected_batch_size}.'
    assert rewards.shape[1] >= len(QRM_HELPSTEER_OBJECTIVES), f'Expected at least 5 objectives, got {rewards.shape[1]}.'
    scores = rewards[:, :len(QRM_HELPSTEER_OBJECTIVES)].numpy()
    assert np.all(np.isfinite(scores)), 'QRM HelpSteer2 scores contain non-finite values.'
    return scores

def tokenize_batch(text_batch):
    encoded = qrm_tokenizer(
        text_batch,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=CONFIG['MAX_LENGTH'],
        add_special_tokens=False,
    )
    return {key: value.to(model_device) for key, value in encoded.items()}



In [ ]:
# Ein einzelnes Sample durchschicken und ALLE Output-Felder inspizieren
with torch.inference_mode():
    probe = qrm_model(**tokenize_batch([chat_texts[0]]))
print('[C-probe] output type:', type(probe))
print('[C-probe] fields:', [k for k in dir(probe) if not k.startswith('_')])
for k in ['score', 'rewards', 'reward_quantiles', 'logits']:
    if hasattr(probe, k):
        v = getattr(probe, k)
        print(f'  {k}: shape={tuple(v.shape)}' if hasattr(v, 'shape') else f'  {k}: {v}')
print('[C-probe] config.attributes:', getattr(qrm_model.config, 'attributes', 'n/a'))


In [ ]:
qrm_scores_chunks = []
qrm_aggregate_chunks = []
reward_quantile_shapes = []
start_time = time.time()
next_progress = 50

for start in range(0, len(chat_texts), CONFIG['BATCH_SIZE']):
    end = min(start + CONFIG['BATCH_SIZE'], len(chat_texts))
    text_batch = chat_texts[start:end]
    encoded = tokenize_batch(text_batch)
    with torch.inference_mode():
        output = qrm_model(**encoded)
    qrm_scores_chunks.append(extract_qrm_helpsteer2_scores(output, expected_batch_size=end - start))
    aggregate = output.score.detach().float().cpu()
    if aggregate.ndim == 1:
        aggregate = aggregate[:, None]
    qrm_aggregate_chunks.append(aggregate.numpy())
    if hasattr(output, 'reward_quantiles') and output.reward_quantiles is not None:
        reward_quantile_shapes.append(tuple(output.reward_quantiles.shape))
    processed = end
    if processed >= next_progress or processed == len(chat_texts):
        elapsed = time.time() - start_time
        print(f'[C] Scored {processed}/{len(chat_texts)} samples in {elapsed:.1f}s')
        while next_progress <= processed:
            next_progress += 50

qrm_scores = np.concatenate(qrm_scores_chunks, axis=0).astype(np.float64)
qrm_aggregate_scores = np.concatenate(qrm_aggregate_chunks, axis=0).astype(np.float64)

assert qrm_scores.shape == (CONFIG['N_SAMPLES'], len(ATTRIBUTES)), qrm_scores.shape
assert qrm_aggregate_scores.shape[0] == CONFIG['N_SAMPLES'], qrm_aggregate_scores.shape
assert np.all(np.isfinite(qrm_scores)), 'QRM scores contain non-finite values.'
assert np.all(np.isfinite(qrm_aggregate_scores)), 'QRM aggregate scores contain non-finite values.'

print('[C] QRM HelpSteer2 objective order:', QRM_HELPSTEER_OBJECTIVES)
print('[C] First reward_quantiles shape seen:', reward_quantile_shapes[0] if reward_quantile_shapes else None)
print('[C] QRM score means:', dict(zip(ATTRIBUTES, np.round(qrm_scores.mean(axis=0), 4))))
print('[C] QRM score stds:', dict(zip(ATTRIBUTES, np.round(qrm_scores.std(axis=0), 4))))

## Correlation structures

Compute Pearson and Spearman matrices for QRM objective scores, HelpSteer2 ground-truth ratings, and QRM-vs-HelpSteer2 cross-axis alignment. All numeric outputs are saved as `.npy` files and printed in a fixed attribute order.

In [ ]:
from scipy.stats import rankdata

MATRIX_FILES = {}

def pairwise_corr(x: np.ndarray, y: np.ndarray | None = None, method: str = 'pearson') -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    y = x if y is None else np.asarray(y, dtype=np.float64)
    assert x.ndim == 2 and y.ndim == 2, 'Correlation inputs must be 2D.'
    out = np.full((x.shape[1], y.shape[1]), np.nan, dtype=np.float64)
    for i in range(x.shape[1]):
        for j in range(y.shape[1]):
            xi = x[:, i]
            yj = y[:, j]
            mask = np.isfinite(xi) & np.isfinite(yj)
            if mask.sum() < 3:
                continue
            xi = xi[mask]
            yj = yj[mask]
            if method == 'spearman':
                xi = rankdata(xi)
                yj = rankdata(yj)
            elif method != 'pearson':
                raise ValueError(f'Unsupported method: {method}')
            if np.std(xi) == 0.0 or np.std(yj) == 0.0:
                continue
            out[i, j] = float(np.corrcoef(xi, yj)[0, 1])
    return out

def save_npy(name: str, array: np.ndarray) -> Path:
    path = OUTPUT_DIR / name
    np.save(path, np.asarray(array))
    assert path.exists(), f'Missing saved file: {path}'
    MATRIX_FILES[name] = path
    return path

def print_matrix(title: str, matrix: np.ndarray, rows=ATTRIBUTES, cols=ATTRIBUTES) -> None:
    print(f'[D] {title}')
    header = ' ' * 14 + ''.join(f'{col[:10]:>12}' for col in cols)
    print(header)
    for row_name, row in zip(rows, matrix):
        values = ''.join(f'{value:12.3f}' if np.isfinite(value) else f'{"nan":>12}' for value in row)
        print(f'{row_name[:12]:>12}  {values}')

qrm_pearson = pairwise_corr(qrm_scores, method='pearson')
qrm_spearman = pairwise_corr(qrm_scores, method='spearman')
hs2_pearson = pairwise_corr(hs2_ratings, method='pearson')
hs2_spearman = pairwise_corr(hs2_ratings, method='spearman')
qrm_vs_hs2_pearson = pairwise_corr(qrm_scores, hs2_ratings, method='pearson')
qrm_vs_hs2_spearman = pairwise_corr(qrm_scores, hs2_ratings, method='spearman')

for name, array in {
    'sample_indices.npy': sample_indices,
    'hs2_ratings.npy': hs2_ratings,
    'qrm_scores.npy': qrm_scores,
    'qrm_aggregate_scores.npy': qrm_aggregate_scores,
    'qrm_pearson.npy': qrm_pearson,
    'qrm_spearman.npy': qrm_spearman,
    'hs2_pearson.npy': hs2_pearson,
    'hs2_spearman.npy': hs2_spearman,
    'qrm_vs_hs2_pearson.npy': qrm_vs_hs2_pearson,
    'qrm_vs_hs2_spearman.npy': qrm_vs_hs2_spearman,
}.items():
    save_npy(name, array)

print_matrix('QRM Pearson: objective scores vs objective scores', qrm_pearson)
print_matrix('QRM Spearman: objective scores vs objective scores', qrm_spearman)
print_matrix('HelpSteer2 Pearson: ground-truth ratings vs ratings', hs2_pearson)
print_matrix('HelpSteer2 Spearman: ground-truth ratings vs ratings', hs2_spearman)
print_matrix('QRM vs HelpSteer2 Pearson: rows=QRM, cols=HS2 ratings', qrm_vs_hs2_pearson)
print_matrix('QRM vs HelpSteer2 Spearman: rows=QRM, cols=HS2 ratings', qrm_vs_hs2_spearman)

for matrix_name, matrix in [
    ('qrm_pearson', qrm_pearson),
    ('qrm_spearman', qrm_spearman),
    ('hs2_pearson', hs2_pearson),
    ('hs2_spearman', hs2_spearman),
    ('qrm_vs_hs2_pearson', qrm_vs_hs2_pearson),
    ('qrm_vs_hs2_spearman', qrm_vs_hs2_spearman),
]:
    assert matrix.shape == (len(ATTRIBUTES), len(ATTRIBUTES)), f'{matrix_name} has wrong shape: {matrix.shape}'

print('[D] Saved matrix files:')
for path in MATRIX_FILES.values():
    print('   ', path)

## Firewall and redundancy check

Document the data-path relationship between QRM, Skywork, HelpSteer2, and ArmoRM. Optional ArmoRM cross-correlations are computed only if a pre-existing score file is supplied; this notebook never calls ArmoRM and never trains.

In [ ]:
FIREWALL_FACTS = {
    'trains_anything': False,
    'uses_armorm_as_training_signal': False,
    'loads_armorm': False,
    'qrm_model': CONFIG['QRM_MODEL'],
    'qrm_backbone_from_model_card': 'Skywork/Skywork-Reward-Llama-3.1-8B',
    'qrm_gating_data_from_model_card': 'Skywork/Skywork-Reward-Preference-80K-v0.1',
    'skywork_dataset_card_contains_helpsteer2_source': True,
    'qrm_predicts_helpsteer_objectives': QRM_HELPSTEER_OBJECTIVES,
    'source_notes': [
        'QRM model card: output.score is aggregate, output.reward_quantiles are reward-distribution quantiles, and first five attributes are HelpSteer2 objectives.',
        'QRM model card: model uses Skywork/Skywork-Reward-Llama-3.1-8B as backbone and Skywork reward preference data for the gating network.',
        'Skywork-Reward-Preference-80K-v0.1 dataset card lists helpsteer2 as a source value.',
    ],
}

assert FIREWALL_FACTS['trains_anything'] is False, 'This notebook must not train.'
assert FIREWALL_FACTS['uses_armorm_as_training_signal'] is False, 'ArmoRM must remain read-only/evaluation-only.'
assert FIREWALL_FACTS['loads_armorm'] is False, 'This pre-check should not load ArmoRM.'
assert CONFIG['QRM_MODEL'] != 'RLHFlow/ArmoRM-Llama3-8B-v0.1', 'Held-out RM must be distinct from ArmoRM.'

armorm_cross_pearson = None
armorm_cross_spearman = None
armorm_score_path = CONFIG.get('ARMORM_SCORE_PATH') or ''
if armorm_score_path:
    armorm_score_path = Path(armorm_score_path).expanduser().resolve()
    if armorm_score_path.exists():
        print('[E] Loading pre-existing ArmoRM scores:', armorm_score_path)
        armorm_scores = np.load(armorm_score_path)
        assert armorm_scores.shape == qrm_scores.shape, f'ArmoRM score shape mismatch: {armorm_scores.shape} vs {qrm_scores.shape}'
        assert np.all(np.isfinite(armorm_scores)), 'ArmoRM scores contain non-finite values.'
        armorm_cross_pearson = pairwise_corr(qrm_scores, armorm_scores, method='pearson')
        armorm_cross_spearman = pairwise_corr(qrm_scores, armorm_scores, method='spearman')
        save_npy('qrm_vs_armorm_pearson.npy', armorm_cross_pearson)
        save_npy('qrm_vs_armorm_spearman.npy', armorm_cross_spearman)
        print_matrix('QRM vs pre-existing ArmoRM Pearson: rows=QRM, cols=ArmoRM', armorm_cross_pearson)
        print_matrix('QRM vs pre-existing ArmoRM Spearman: rows=QRM, cols=ArmoRM', armorm_cross_spearman)
    else:
        print('[E] ARMORM_SCORE_PATH was set but does not exist; cross-correlation not measured:', armorm_score_path)
else:
    print('[E] ArmoRM cross-correlation not measured. Set CONFIG["ARMORM_SCORE_PATH"] to a saved N x 5 .npy file to enable it.')

firewall_path = OUTPUT_DIR / 'firewall_redundancy_check.json'
with firewall_path.open('w', encoding='utf-8') as handle:
    json.dump(FIREWALL_FACTS, handle, indent=2, sort_keys=True)
print('[E] Saved firewall/redundancy notes:', firewall_path)
print('[E] Key caution: QRM is distinct from ArmoRM, but it is not data-independent from HelpSteer2 because its published objective set and Skywork data path include HelpSteer2.')

## Verdict

Turn the measured correlations into a clear GO/STOP pre-check. The main decision is whether QRM's help/correctness Pearson correlation is below the configured decorrelation threshold.

In [ ]:
try:
    import pandas as pd
except Exception:
    pd = None

help_idx = ATTRIBUTES.index('helpfulness')
corr_idx = ATTRIBUTES.index('correctness')

r_help_corr_qrm = float(qrm_pearson[help_idx, corr_idx])
r_help_corr_hs2_sample = float(hs2_pearson[help_idx, corr_idx])
r_help_corr_reference = float(CONFIG['HS2_HELP_CORR_REFERENCE'])
threshold = float(CONFIG['DECORR_THRESHOLD'])

if abs(r_help_corr_qrm) < threshold:
    decision = 'GO'
    decision_text = (
        f'GO: QRM decorrelates help/correctness noticeably '
        f'(r={r_help_corr_qrm:.3f} < {threshold:.3f}); held-out PPO pilot is potentially meaningful.'
    )
else:
    decision = 'STOP'
    decision_text = (
        f'STOP: QRM inherits help/correctness collinearity '
        f'(r={r_help_corr_qrm:.3f} >= {threshold:.3f}); PPO against QRM would likely keep Wand B unsolved.'
    )

pair_rows = []
negative_or_weak_pairs = []
strong_positive_pairs = []
for i, left in enumerate(ATTRIBUTES):
    for j in range(i + 1, len(ATTRIBUTES)):
        right = ATTRIBUTES[j]
        qrm_r = float(qrm_pearson[i, j])
        hs2_r = float(hs2_pearson[i, j])
        if not np.isfinite(qrm_r):
            label = 'nan'
        elif qrm_r < 0.0:
            label = 'negative'
            negative_or_weak_pairs.append((left, right, qrm_r))
        elif abs(qrm_r) < 0.3:
            label = 'weak'
            negative_or_weak_pairs.append((left, right, qrm_r))
        elif qrm_r >= threshold:
            label = 'strong_positive'
            strong_positive_pairs.append((left, right, qrm_r))
        else:
            label = 'moderate_positive'
        pair_rows.append({
            'axis_pair': f'{left}__{right}',
            'qrm_pearson': qrm_r,
            'hs2_pearson_sample': hs2_r,
            'qrm_minus_hs2_sample': qrm_r - hs2_r,
            'qrm_label': label,
        })

verdict = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'decision_text': decision_text,
    'qrm_model': CONFIG['QRM_MODEL'],
    'n_samples': int(CONFIG['N_SAMPLES']),
    'hs2_split': CONFIG['HS2_SPLIT'],
    'seed': int(CONFIG['SEED']),
    'decorrelation_threshold': threshold,
    'documented_help_corr_reference': r_help_corr_reference,
    'measured_help_corr_hs2_sample': r_help_corr_hs2_sample,
    'measured_help_corr_qrm': r_help_corr_qrm,
    'negative_or_weak_qrm_pairs': [
        {'left': left, 'right': right, 'qrm_pearson': value}
        for left, right, value in negative_or_weak_pairs
    ],
    'strong_positive_qrm_pairs': [
        {'left': left, 'right': right, 'qrm_pearson': value}
        for left, right, value in strong_positive_pairs
    ],
    'pair_table': pair_rows,
    'armorm_cross_correlation_measured': armorm_cross_pearson is not None,
}

verdict_path = OUTPUT_DIR / 'verdict.json'
with verdict_path.open('w', encoding='utf-8') as handle:
    json.dump(verdict, handle, indent=2, sort_keys=True)

print('[F]', decision_text)
print(f'[F] Reference full-data HelpSteer2 help/correctness r documented in project: {r_help_corr_reference:.3f}')
print(f'[F] This sampled HelpSteer2 help/correctness r: {r_help_corr_hs2_sample:.3f}')
print('[F] Potential conflict structure from QRM negative/weak pairs:')
if negative_or_weak_pairs:
    for left, right, value in negative_or_weak_pairs:
        print(f'    {left} vs {right}: r={value:.3f}')
else:
    print('    none')
print('[F] Strong positive QRM pairs:')
if strong_positive_pairs:
    for left, right, value in strong_positive_pairs:
        print(f'    {left} vs {right}: r={value:.3f}')
else:
    print('    none')

if pd is not None:
    pair_df = pd.DataFrame(pair_rows)
    display(pair_df)
    pair_df.to_csv(OUTPUT_DIR / 'axis_pair_summary.csv', index=False)
else:
    print('[F] Pair table:')
    for row in pair_rows:
        print(row)

assert verdict_path.exists(), 'verdict.json was not written.'
print('[F] Saved verdict:', verdict_path)

## Report and output zip

Write a compact Markdown report, bundle all saved matrices plus the verdict into a zip file, and download it from Colab. Generated outputs should stay out of GitHub unless you explicitly decide otherwise.

In [ ]:
report_path = OUTPUT_DIR / 'qrm_precheck_summary.md'
zip_path = OUTPUT_DIR / CONFIG['OUTPUT_ZIP']

pair_lines = []
for row in verdict['pair_table']:
    pair_lines.append(
        f"| {row['axis_pair']} | {row['qrm_pearson']:.3f} | {row['hs2_pearson_sample']:.3f} | {row['qrm_label']} |"
    )

report_text = f"""# QRM axis decorrelation pre-check

Created UTC: {verdict['created_at_utc']}

## Configuration

```json
{json.dumps(CONFIG, indent=2, sort_keys=True)}
```

## Decision

{verdict['decision_text']}

- QRM help/correctness Pearson: {verdict['measured_help_corr_qrm']:.3f}
- Sample HelpSteer2 help/correctness Pearson: {verdict['measured_help_corr_hs2_sample']:.3f}
- Documented project reference for HelpSteer2 help/correctness Pearson: {verdict['documented_help_corr_reference']:.3f}
- Decorrelation threshold: {verdict['decorrelation_threshold']:.3f}

## Axis-pair table

| pair | QRM Pearson | HelpSteer2 sample Pearson | QRM label |
|---|---:|---:|---|
{chr(10).join(pair_lines)}

## Firewall notes

- This notebook trains nothing.
- ArmoRM is not loaded and is not used as a training signal.
- QRM is a distinct model from ArmoRM, but the published QRM/Skywork data path is not independent of HelpSteer2.
- Optional QRM-vs-ArmoRM cross-correlation measured: {verdict['armorm_cross_correlation_measured']}.

## Interpretation

GO means QRM may be worth a small PPO pilot because it weakens the help/correctness coupling. STOP means QRM likely inherits the same co-rating structure and should not be used for a full PPO path without a different held-out RM or another setup change.
"""

report_path.write_text(report_text, encoding='utf-8')
assert report_path.exists(), 'Summary report was not written.'

files_to_zip = [
    OUTPUT_DIR / 'sample_records.jsonl',
    OUTPUT_DIR / 'firewall_redundancy_check.json',
    OUTPUT_DIR / 'verdict.json',
    report_path,
]
files_to_zip.extend(MATRIX_FILES.values())
axis_pair_csv = OUTPUT_DIR / 'axis_pair_summary.csv'
if axis_pair_csv.exists():
    files_to_zip.append(axis_pair_csv)

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files_to_zip:
        if path.exists():
            archive.write(path, arcname=path.name)
            print('[G] Added to zip:', path.name)
        else:
            print('[G] Skipped missing file:', path)

assert zip_path.exists(), 'Output zip was not created.'
print('[G] Output zip:', zip_path)
print('[G] Zip size MB:', round(zip_path.stat().st_size / (1024 * 1024), 3))
print('[G] Keep generated outputs out of Git unless you intentionally want to version a specific result snapshot.')

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as error:
    print('[G] files.download is available only in Colab. Download manually from:', zip_path)
    print('[G] Download error:', repr(error))